# Kandinsky V2A — Speed and Duration Pilot

## Goal

Measure whether the current **Full Lite video-to-audio workflow** can run faster or generate beyond its nominal five-second clip length, while keeping the five-second result as a quality reference.

This is an inference experiment only. It does not train or fine-tune the model.

## Controlled comparison

All runs use the same source-video start, prompt, seed, Full Lite checkpoint, and output resolution.

1. **Baseline:** 5 seconds, PyTorch SDPA attention, no compilation.
2. **Compilation candidate:** 5 seconds, same settings with `torch.compile`.
3. **Optional attention candidate:** 5 seconds with an already-installed GPU-compatible attention kernel. The notebook skips this when the required package is absent; it does not install risky CUDA extensions.
4. **Duration extension:** 6 seconds, baseline settings. This is exploratory because the published Lite checkpoint is a five-second model.

The 6-second test is deliberate: this code path takes integer seconds, and a one-second extension is a safer first extrapolation than jumping straight to 7.5 or 15 seconds.

## Architecture notes relevant to this experiment

Kandinsky 6 uses a **dual-stream CrossDiT**: a video stream and an audio stream exchange information through bidirectional cross-attention. In video-to-audio mode, the source video is encoded as visual conditioning, audio latents are generated, then the audio VAE and vocoder decode the waveform. The input video frames are retained in the output container. The model is therefore doing cross-modal audio generation, not merely attaching an unrelated audio file.

For Full Lite, the checkpoint config specifies 31 latent video frames / 121 sampled frames for its five-second setting. The pipeline computes a requested duration as `time_length * 24 // 4 + 1`: five seconds gives 31 temporal latents, while six gives 37. Longer duration increases the joint sequence workload; attention and memory may grow faster than the duration ratio. This notebook measures it instead of assuming linear scaling.

The pipeline explicitly requires `cache.mode: none` for video/audio conditioning. MagCache and NaviCache are therefore not valid speed switches for this V2A path in the pinned implementation. The practical inference knobs we can compare here are compilation and a compatible attention kernel. Lowering resolution or denoising steps may also reduce work, but those can change quality and are intentionally held fixed for this first comparison.

**Interpretation limit:** Kandinsky's published timing table is for text-to-audio-video generation and excludes weight loading and MP4 encoding. These notebook timings include the full CLI call (model load, generation, and save), so compare runs in this notebook with one another, not directly with the published numbers.

Sources: [Kandinsky 6 paper](https://arxiv.org/abs/2610.05608), [official pinned model config](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/configs/checkpoints/lite.yaml), [pipeline implementation](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/pipeline/pipeline.py), and [generation modes](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/docs/generation_modes.md).

## 1 — Check the Colab runtime and choose storage

A100 80 GB is a good target for Full Lite, but Colab's notebook metadata cannot force a particular GPU. This cell reports the GPU actually assigned. Use a GPU runtime and keep at least 35 GiB of free disk for model weights, caches, and generated files.

In [ ]:
import os, sys, shutil, subprocess, json, time, hashlib, zipfile, re
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import Audio, Video, display, Markdown

if not shutil.which("nvidia-smi"):
    raise RuntimeError("No NVIDIA GPU is visible. In Colab, choose Runtime > Change runtime type > GPU, then reconnect.")

def query_gpu():
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True
    )
    if result.returncode:
        raise RuntimeError("Could not query the GPU: " + result.stderr[-1000:])
    return result.stdout.strip().splitlines()[0]

GPU_DESCRIPTION = query_gpu()
print("Python:", sys.version.split()[0])
print("Assigned GPU:", GPU_DESCRIPTION)

roots = [Path("/content/local-scratch"), Path("/content")]
available_roots = [p for p in roots if p.exists()]
CACHE_ROOT = max(available_roots, key=lambda p: shutil.disk_usage(p).free)
free_gib = shutil.disk_usage(CACHE_ROOT).free / (1024 ** 3)
print(f"Model/cache storage: {CACHE_ROOT} | free space: {free_gib:.1f} GiB")
if free_gib < 35:
    raise RuntimeError("Free at least 35 GiB of disk space or select a larger Colab runtime.")

os.environ["HF_HOME"] = str(CACHE_ROOT / "huggingface")
os.environ["HUGGINGFACE_HUB_CACHE"] = str(CACHE_ROOT / "huggingface" / "hub")
os.environ["KANDINSKY_HOME"] = str(CACHE_ROOT / "kandinsky_home")

WORK = Path("/content/kandinsky_v2a_speed_duration")
INPUT_DIR = WORK / "input"
WINDOW_DIR = WORK / "conditions"
OUTPUT_DIR = WORK / "outputs"
LOG_DIR = WORK / "logs"
for directory in (INPUT_DIR, WINDOW_DIR, OUTPUT_DIR, LOG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print("Experiment files:", WORK)

## 2 — Install the same pinned Kandinsky source used by the current Full Lite notebook

The source commit is pinned so a later upstream change cannot silently alter the experiment. This setup uses the official consumer dependency set and does not compile or install optional attention packages.

In [ ]:
REPO = Path("/content/kandinsky-6-speed-study")
PINNED_COMMIT = "7ceeefad7fb625011f7db602c43893a989f19c15"

if not (REPO / ".git").exists():
    subprocess.run(
        ["git", "clone", "--filter=blob:none", "https://github.com/kandinskylab/kandinsky-6.git", str(REPO)],
        check=True
    )
subprocess.run(["git", "-C", str(REPO), "checkout", PINNED_COMMIT], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
if not shutil.which("uv"):
    raise RuntimeError("The uv installer completed, but the uv command is not on PATH. Restart the runtime and rerun this cell.")

if not shutil.which("just"):
    install_just = subprocess.run(
        ["bash", "-lc", "curl -fsSL https://just.systems/install.sh | bash -s -- --to /usr/local/bin"],
        capture_output=True, text=True
    )
    if install_just.returncode:
        print(install_just.stdout[-2000:])
        print(install_just.stderr[-3000:])
        raise RuntimeError("Could not install just; see the installer output above.")

setup = subprocess.run(
    ["uv", "sync", "--extra", "consumer", "--inexact", "--no-dev"],
    cwd=REPO, capture_output=True, text=True
)
print(setup.stdout[-4000:])
if setup.returncode:
    print(setup.stderr[-6000:])
    raise RuntimeError("Kandinsky dependency setup failed. The underlying error is shown above.")

SOURCE_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip()
PY = REPO / ".venv" / "bin" / "python"
if not PY.is_file():
    raise FileNotFoundError(f"Kandinsky virtual-environment Python is missing: {PY}")

# Report whether this runtime already has the optional acceleration backends.
backend_check = subprocess.run(
    [str(PY), "-c",
     "import importlib.util,json; print(json.dumps({"
     "'sageattention': importlib.util.find_spec('sageattention') is not None,"
     "'flash_attn': importlib.util.find_spec('flash_attn') is not None,"
     "'flash_attn_interface': importlib.util.find_spec('flash_attn_interface') is not None}))"],
    cwd=REPO, capture_output=True, text=True
)
if backend_check.returncode:
    print(backend_check.stderr[-2000:])
    raise RuntimeError("Could not inspect optional attention packages.")
AVAILABLE_BACKENDS = json.loads(backend_check.stdout.strip())

print("Kandinsky source commit:", SOURCE_COMMIT)
print("Optional attention packages already present:", AVAILABLE_BACKENDS)
print("Downloading Full Lite weights (only needed once per Colab cache)...")
download = subprocess.run(["just", "download", "lite"], cwd=REPO, text=True)
if download.returncode:
    raise RuntimeError("Full Lite checkpoint download failed.")
print("Pinned environment and Full Lite checkpoint are ready.")

## 3 — Select the video segment, prompt, and seed

Use the same video segment and exact prompt as the last successful Full Lite comparison. The default prompt below preserves the text you used. If your uploaded file is already cut to the intended segment, leave `SOURCE_START_SECONDS = 0`. If it is the full original video and you want the test to begin at 15 seconds, set it to 15.

The source must contain at least 6.25 seconds from that start. A small extra tail ensures the sampler has enough 24 fps frames for its integer six-second request; the model trims the condition to the requested duration.

In [ ]:
DEFAULT_VIDEO = Path("/content/kandinsky_v2a_input.mp4")
SOURCE_VIDEO = DEFAULT_VIDEO
if not SOURCE_VIDEO.is_file():
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No video was uploaded. Run this cell again and select the same source video.")
    uploaded_name = next(iter(uploaded))
    SOURCE_VIDEO = Path("/content") / Path(uploaded_name).name
    SOURCE_VIDEO.write_bytes(uploaded[uploaded_name])

if not SOURCE_VIDEO.is_file():
    raise FileNotFoundError(f"Source video not found: {SOURCE_VIDEO}")

# Keep these identical across runs and match your previous successful test.
PROMPT = "Generate natural sound that matches the visible events in this video. woman fbi agent wallking in forest with her gun. No music. No speech."
SEED = 42
SOURCE_START_SECONDS = 0  # Set to 15 only if SOURCE_VIDEO is the untrimmed original.
TEST_DURATIONS_SECONDS = [5, 6]

def run_capture(command, cwd=None):
    result = subprocess.run(command, cwd=cwd, capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError((result.stderr or result.stdout)[-5000:])
    return result.stdout.strip()

probe = json.loads(run_capture([
    "ffprobe", "-v", "error", "-show_streams", "-show_format", "-of", "json", str(SOURCE_VIDEO)
]))
video_streams = [s for s in probe.get("streams", []) if s.get("codec_type") == "video"]
if not video_streams:
    raise ValueError("The selected file has no video stream.")
SOURCE_DURATION = float(probe["format"]["duration"])
required_duration = SOURCE_START_SECONDS + max(TEST_DURATIONS_SECONDS) + 0.25
if SOURCE_DURATION < required_duration:
    raise ValueError(
        f"Need at least {required_duration:.2f}s from the selected start; "
        f"this file is {SOURCE_DURATION:.2f}s. Choose the correct source segment or start time."
    )

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

print(f"Source: {SOURCE_VIDEO.name} | duration {SOURCE_DURATION:.2f}s")
print("Video stream:", video_streams[0].get("codec_name"), video_streams[0].get("width"), "x", video_streams[0].get("height"))
print("Source SHA-256:", sha256_file(SOURCE_VIDEO))
print("Prompt:", PROMPT)
print("Seed:", SEED, "| source start:", SOURCE_START_SECONDS, "seconds")
display(Video(filename=str(SOURCE_VIDEO), embed=True, width=800))

## 4 — Prepare matching 5-second and 6-second video conditions

Both conditions begin at the same timestamp and use the same 24 fps / resize / encoding path. Audio from the source file is removed. The extra 0.25 seconds is available to the loader, which uses the requested `time_length` and trims longer video conditions.

In [ ]:
CONDITIONS = {}
for duration in TEST_DURATIONS_SECONDS:
    condition_path = WINDOW_DIR / f"condition_{duration}s.mp4"
    input_tail = duration + 0.25
    command = [
        "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
        "-ss", str(SOURCE_START_SECONDS), "-i", str(SOURCE_VIDEO),
        "-t", str(input_tail), "-map", "0:v:0", "-an",
        "-vf", "fps=24,scale='min(960,iw)':-2",
        "-c:v", "libx264", "-preset", "veryfast", "-crf", "25",
        "-pix_fmt", "yuv420p", str(condition_path)
    ]
    run_capture(command)
    if not condition_path.is_file():
        raise FileNotFoundError(f"FFmpeg did not create {condition_path}")
    condition_probe = json.loads(run_capture([
        "ffprobe", "-v", "error", "-show_entries", "format=duration",
        "-of", "json", str(condition_path)
    ]))
    actual_condition_duration = float(condition_probe["format"]["duration"])
    CONDITIONS[duration] = {
        "path": condition_path,
        "duration_seconds": actual_condition_duration,
        "sha256": sha256_file(condition_path),
    }
    print(
        f"{duration}s request condition: {condition_path.name} | "
        f"available {actual_condition_duration:.2f}s | SHA-256 {CONDITIONS[duration]['sha256']}"
    )

## 5 — Review the exact conditions before spending GPU time

Confirm the two videos show the same event and start at the same point. The six-second file should only provide a little extra tail for safe frame sampling.

In [ ]:
for duration in TEST_DURATIONS_SECONDS:
    item = CONDITIONS[duration]
    display(Markdown(f"### Condition for a {duration}-second generation"))
    display(Video(filename=str(item["path"]), embed=True, width=700))

## 6 — Configure the fixed generation settings

These YAML files keep the Full Lite checkpoint, resolution, denoising steps, cache mode, offload, and text rewriting fixed. The baseline uses SDPA and no compilation. The compile candidate changes only `compile.strategy`. The prompt beautifier stays enabled as in the prior Full Lite workflow; the expanded prompt saved by Kandinsky is recorded with each run.

The video-to-audio pipeline rejects MagCache/NaviCache settings, so this experiment deliberately keeps cache mode disabled.

In [ ]:
RUN_TORCH_COMPILE = True
RUN_OPTIONAL_ATTENTION = True
RUN_SIX_SECOND_EXTENSION = True
BEAUTIFIER = "qwen25"  # Keep the same prompt-rewrite behavior throughout this pilot.

def write_device_config(name, compile_strategy="none", attention="sdpa"):
    config_path = WORK / f"{name}.yaml"
    config_path.write_text(
        f"""checkpoint: lite
cache:
  mode: none
offload:
  strategy: none
compile:
  strategy: {compile_strategy}
attention:
  engine: {attention}
text_embedder:
  quantized_qwen: false
beautifier:
  name: {BEAUTIFIER}
""",
        encoding="utf-8",
    )
    return config_path

print("Controls:", {
    "checkpoint": "lite (Full Lite)",
    "compile": "none for baseline",
    "attention": "sdpa for baseline",
    "cache": "none (required for V2A conditioning)",
    "beautifier": BEAUTIFIER,
    "prompt": PROMPT,
    "seed": SEED,
})

## 7 — Run one generation at a time and sample resource use

The measured wall time covers the same end-to-end CLI route used by the notebook: model startup/weight load, denoising, decoding, and MP4 save. It is not a pure denoising-kernel benchmark. GPU memory/utilization are sampled every two seconds, so a very brief peak could be missed.

The optional attention test runs only if a matching package is already installed in the pinned Kandinsky environment. No attention package is installed by this notebook.

In [ ]:
def gpu_sample():
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.used,memory.total,utilization.gpu",
         "--format=csv,noheader,nounits"],
        capture_output=True, text=True
    )
    if result.returncode:
        return None
    try:
        used, total, util = [int(x.strip()) for x in result.stdout.strip().split(",")]
        return {"memory_used_mib": used, "memory_total_mib": total, "utilization_gpu_percent": util}
    except Exception:
        return None

def run_generation(run_id, duration, compile_strategy="none", attention="sdpa"):
    condition = CONDITIONS[duration]
    config_path = write_device_config(run_id, compile_strategy, attention)
    output_mp4 = OUTPUT_DIR / f"{run_id}.mp4"
    output_wav = OUTPUT_DIR / f"{run_id}.wav"
    log_path = LOG_DIR / f"{run_id}.log"
    command = [
        "just", "generate", PROMPT,
        "--config", str(config_path),
        "--attention-engine", attention,
        "--video-file", str(condition["path"]),
        "--time-length", str(duration),
        "--out", str(output_mp4),
        "--seed", str(SEED),
    ]
    print(f"\nStarting {run_id}: {duration}s | compile={compile_strategy} | attention={attention}")
    samples = []
    started = time.time()
    with open(log_path, "w", encoding="utf-8") as log:
        process = subprocess.Popen(command, cwd=REPO, stdout=log, stderr=subprocess.STDOUT)
        while process.poll() is None:
            sample = gpu_sample()
            if sample:
                samples.append(sample)
            time.sleep(2)
        return_code = process.returncode
    final_sample = gpu_sample()
    if final_sample:
        samples.append(final_sample)
    elapsed = round(time.time() - started, 2)
    log_text = log_path.read_text(encoding="utf-8", errors="replace")
    if return_code != 0:
        print(log_text[-5000:])
        return {
            "run_id": run_id, "status": "failed", "duration_seconds": duration,
            "compile_strategy": compile_strategy, "attention": attention,
            "elapsed_seconds": elapsed, "log_file": log_path.name,
            "error_tail": log_text[-5000:],
        }
    if not output_mp4.is_file():
        return {
            "run_id": run_id, "status": "failed", "duration_seconds": duration,
            "compile_strategy": compile_strategy, "attention": attention,
            "elapsed_seconds": elapsed, "log_file": log_path.name,
            "error_tail": "CLI returned success but the expected MP4 is missing.",
        }

    extract = subprocess.run([
        "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
        "-i", str(output_mp4), "-vn", "-ac", "2", "-ar", "44100",
        "-c:a", "pcm_s16le", str(output_wav)
    ], capture_output=True, text=True)
    if extract.returncode:
        print(extract.stderr[-3000:])
        return {
            "run_id": run_id, "status": "failed_audio_extract", "duration_seconds": duration,
            "compile_strategy": compile_strategy, "attention": attention,
            "elapsed_seconds": elapsed, "log_file": log_path.name,
            "error_tail": extract.stderr[-3000:],
        }

    output_probe = json.loads(run_capture([
        "ffprobe", "-v", "error", "-show_entries", "format=duration",
        "-of", "json", str(output_mp4)
    ]))
    expanded_prompt_path = output_mp4.with_suffix(".txt")
    result = {
        "run_id": run_id,
        "status": "ok",
        "duration_request_seconds": duration,
        "output_duration_seconds": float(output_probe["format"]["duration"]),
        "compile_strategy": compile_strategy,
        "attention": attention,
        "elapsed_seconds": elapsed,
        "gpu_memory_peak_sampled_mib": max((x["memory_used_mib"] for x in samples), default=None),
        "gpu_utilization_peak_sampled_percent": max((x["utilization_gpu_percent"] for x in samples), default=None),
        "input_condition_sha256": condition["sha256"],
        "output_mp4": output_mp4.name,
        "output_wav": output_wav.name,
        "output_mp4_sha256": sha256_file(output_mp4),
        "output_wav_sha256": sha256_file(output_wav),
        "expanded_prompt": expanded_prompt_path.read_text(encoding="utf-8").strip() if expanded_prompt_path.is_file() else None,
        "log_file": log_path.name,
    }
    print(f"Completed {run_id} in {elapsed:.1f}s")
    if result["gpu_memory_peak_sampled_mib"] is not None:
        print("Peak sampled GPU memory:", result["gpu_memory_peak_sampled_mib"], "MiB")
    return result

RUNS = []
# The baseline is the reference point. If it fails, its log is still included in the report.
RUNS.append(run_generation("full_lite_5s_sdpa_baseline", 5, "none", "sdpa"))

if RUN_TORCH_COMPILE:
    RUNS.append(run_generation("full_lite_5s_torch_compile", 5, "torch", "sdpa"))

if RUN_OPTIONAL_ATTENTION:
    gpu_name_lower = GPU_DESCRIPTION.lower()
    if "h100" in gpu_name_lower and AVAILABLE_BACKENDS.get("flash_attn_interface"):
        RUNS.append(run_generation("full_lite_5s_flash_attention_3", 5, "none", "flash_attention_3"))
    elif AVAILABLE_BACKENDS.get("sageattention"):
        RUNS.append(run_generation("full_lite_5s_sage_attention", 5, "none", "sage"))
    elif AVAILABLE_BACKENDS.get("flash_attn"):
        RUNS.append(run_generation("full_lite_5s_flash_attention_2", 5, "none", "flash_attention_2"))
    else:
        print("\nOptional attention comparison skipped: no known compatible package is already installed.")

if RUN_SIX_SECOND_EXTENSION:
    RUNS.append(run_generation("full_lite_6s_extension", 6, "none", "sdpa"))

print("\nRun status summary:")
for item in RUNS:
    print(item["run_id"], "—", item["status"], "| elapsed:", item.get("elapsed_seconds"), "s")

## 8 — Listen and score the successful outputs

Compare each generated result with its own video condition. For speed, compare wall time and sampled GPU memory. For the six-second run, decide whether the extra second remains coherent and whether audio quality, event match, and synchronization hold up against the five-second reference.

A successful MP4 or audio extraction is not evidence that the audio sounds good; add your own listening notes below.

In [ ]:
SUCCESSFUL_RUNS = [item for item in RUNS if item.get("status") == "ok"]
LISTENING_NOTES = {
    item["run_id"]: {
        "audio_quality_1_to_5": None,
        "noise_or_artifacts_1_to_5": None,  # 1 = very low, 5 = very prominent
        "event_match_1_to_5": None,
        "sync_1_to_5": None,
        "six_second_continuity_1_to_5": None,
        "notes": "",
    }
    for item in SUCCESSFUL_RUNS
}

for item in SUCCESSFUL_RUNS:
    duration = item["duration_request_seconds"]
    display(Markdown(
        f"### {item['run_id']} — requested {duration}s, generated in {item['elapsed_seconds']:.0f}s"
    ))
    display(Video(
        filename=str(OUTPUT_DIR / item["output_mp4"]),
        embed=True, width=800
    ))
    display(Audio(filename=str(OUTPUT_DIR / item["output_wav"])))
print("\nFill LISTENING_NOTES in the next cell after listening.")

In [ ]:
# Enter scores after listening. Keep None for anything you have not judged yet.
# Example:
# LISTENING_NOTES["full_lite_5s_sdpa_baseline"]["audio_quality_1_to_5"] = 4
# LISTENING_NOTES["full_lite_5s_sdpa_baseline"]["notes"] = "Clear event match; mild background hiss."

print(json.dumps(LISTENING_NOTES, ensure_ascii=False, indent=2))

## 9 — Save the reproducible report and downloadable outputs

The ZIP contains the experiment metadata, generated MP4/WAV files, and CLI logs. It does not contain the source video. Use the JSON to compare results later on AWS or after changing the serving path.

In [ ]:
REPORT = {
    "experiment": "kandinsky_v2a_speed_duration_pilot_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_video_name": SOURCE_VIDEO.name,
    "source_video_sha256": sha256_file(SOURCE_VIDEO),
    "source_video_duration_seconds": SOURCE_DURATION,
    "source_start_seconds": SOURCE_START_SECONDS,
    "source_commit": SOURCE_COMMIT,
    "gpu": GPU_DESCRIPTION,
    "checkpoint": "kandinskylab/Kandinsky-6.0-Lite-5s-Diffusers",
    "prompt": PROMPT,
    "seed": SEED,
    "beautifier": BEAUTIFIER,
    "conditions": {
        str(k): {
            "file": v["path"].name,
            "available_duration_seconds": v["duration_seconds"],
            "sha256": v["sha256"],
        } for k, v in CONDITIONS.items()
    },
    "runs": RUNS,
    "listening_notes": LISTENING_NOTES,
    "limitations": [
        "The Lite checkpoint is published as a five-second model; six seconds is an extrapolation experiment.",
        "GPU memory and utilization are sampled every two seconds, not collected with a device-side peak-memory counter.",
        "Wall time includes model loading, generation, audio/video decoding, and output writing.",
        "No training, fine-tuning, cache acceleration, or source-video upload is included in the result ZIP.",
    ],
}
REPORT_PATH = WORK / "speed_duration_report.json"
REPORT_PATH.write_text(json.dumps(REPORT, ensure_ascii=False, indent=2), encoding="utf-8")

ZIP_PATH = Path("/content/kandinsky_v2a_speed_duration_report.zip")
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write(REPORT_PATH, "speed_duration_report.json")
    for item in RUNS:
        for key in ("output_mp4", "output_wav", "log_file"):
            name = item.get(key)
            if not name:
                continue
            source = (LOG_DIR / name) if key == "log_file" else (OUTPUT_DIR / name)
            if source.is_file():
                archive.write(source, f"{key.removesuffix('_file')}s/{source.name}" if key != "log_file" else f"logs/{source.name}")
    for duration, condition in CONDITIONS.items():
        archive.write(condition["path"], f"conditions/{condition['path'].name}")

print("Report:", REPORT_PATH)
print("Download ZIP:", ZIP_PATH, "|", round(ZIP_PATH.stat().st_size / 1e6, 1), "MB")
from google.colab import files
files.download(str(ZIP_PATH))